In [35]:
# Imports 
import re
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
# Load data and drop null values 
df = pd.read_csv("../data/raw/training_data.csv")
print(df.isna().sum())
df = df.dropna(subset=["text"])

label    0
text     2
dtype: int64


In [3]:
# clean and tokanize
pattern = re.compile(r'[^a-zA-Z\s]+')
noise_pattern = re.compile(r'escapenumber|escapelong', re.IGNORECASE)
stopwords = {'y', 'did', 'i', 'again', "hadn't", 'my', 'over', 'too', 'here', "that'll", 'couldn', 'than', "they've", 'same', "she's", 'they', 'doing', 'if', 'down', "don't", 'out', 'no', 'her', 're', 'such', 't', "i've", 'only', 'was', 'shouldn', "they'd", 'won', 'more', 'needn', 'do', "shouldn't", "wasn't", "aren't", 'aren', 'how', 'shan', 'doesn', 'few', 'll', 'myself', "couldn't", "he'd", 'other', 'wasn', 'his', 'on', 'these', 'both', "didn't", 'you', "you've", 'their', 'what', "you'd", 'being', 'by', 'been', 's', 'ain', 'why', 'where', 'until', 'as', 'off', 'after', 'is', 'be', 'below', 'hasn', 'yours', 'we', 'has', 'own', 've', 'haven', 'whom', 'wouldn', 'hers', "we'd", 'between', 'o', "he's", 'have', 'herself', 'does', 'now', "shan't", 'don', 'in', 'so', 'from', 'a', 'under', 'further', 'those', 'me', 'most', "weren't", 'against', 'its', 'she', 'at', "you'll", 'yourself', "i'll", 'm', 'once', 'mustn', 'while', 'should', 'ours', 'didn', 'then', 'when', 'that', 'were', "it'd", 'which', 'above', 'all', "doesn't", "mustn't", "she'd", "it's", 'before', 'of', 'and', 'weren', "she'll", 'our', 'will', 'isn', "i'm", 'had', 'to', 'the', 'through', 'with', 'there', 'during', 'or', "haven't", 'himself', 'it', 'theirs', "wouldn't", 'each', 'not', 'just', 'this', "we'll", "he'll", "needn't", "we've", 'ourselves', 'about', "isn't", 'your', 'nor', 'because', 'can', 'he', 'am', "i'd", "should've", 'any', 'd', 'some', 'having', 'ma', 'itself', 'into', "we're", "hasn't", "they'll", 'who', 'are', 'but', 'themselves', "mightn't", "it'll", 'very', 'for', 'hadn', "you're", 'him', 'them', 'an', "they're", 'mightn', "won't", 'yourselves', 'up'}
NOISE_WORDS = {"subject", "pm", "org", "com", "http", "www"}

def clean(email: str) -> list[str]:
    email = noise_pattern.sub('', email)
    words = email.lower().split()
    words = [pattern.sub('', w) for w in words]
    words = [
        w for w in words
        if w and len(w) > 1 and w not in stopwords and w not in NOISE_WORDS
    ]
    return words

In [ ]:
# build vocabulary 
spam_column = df[df["label"] == "Spam"]
spam_text_column = spam_column["text"]

ham_column = df[df["label"] == "Ham"]
ham_text_column = ham_column["text"]


spam_vocab = []
ham_vocab = []

for ham_email in ham_text_column:
    ham_vocab.extend(clean(ham_email))

for spam_email in spam_text_column:
    spam_vocab.extend(clean(spam_email))

global_vocab = set(ham_vocab) | set(spam_vocab)

In [19]:
V = len(global_vocab)
indexed_vocab = {word: i for i, word in enumerate(global_vocab)}
indexed_vocab

{'featureful': 0,
 'reiss': 1,
 'nagenoeg': 2,
 'triarchy': 3,
 'prorities': 4,
 'mfiz': 5,
 'resinblzsd': 6,
 'ftiisgusdib': 7,
 'cherri': 8,
 'exaltabuntur': 9,
 'rids': 10,
 'discountair': 11,
 'ldxjc': 12,
 'podanej': 13,
 'shimko': 14,
 'lirefresh': 15,
 'mprvartonumber': 16,
 'freestatistics': 17,
 'settingszoologymy': 18,
 'insinkerator': 19,
 'provisolockout': 20,
 'cursestyp': 21,
 'kindled': 22,
 'eived': 23,
 'gjbiyc': 24,
 'interlays': 25,
 'pragmatists': 26,
 'offloads': 27,
 'psfggh': 28,
 'ewqhppuqkfs': 29,
 'jbuta': 30,
 'slooooowwwly': 31,
 'vajathube': 32,
 'cerise': 33,
 'faby': 34,
 'etejjc': 35,
 'nvcvtx': 36,
 'etwaige': 37,
 'permissive': 38,
 'waake': 39,
 'transoak': 40,
 'eell': 41,
 'urgz': 42,
 'dimacs': 43,
 'anden': 44,
 'coteux': 45,
 'setlength': 46,
 'inteliectua': 47,
 'gravitating': 48,
 'ecologically': 49,
 'incoterms': 50,
 'serveu': 51,
 'characterised': 52,
 'meditation': 53,
 'patts': 54,
 'canedy': 55,
 'simlock': 56,
 'edelston': 57,
 'revender

In [29]:
# Compute priors 
corpus = len(df)
prior_spam = np.log(len(spam_column) / corpus) # log P(spam)
prior_ham = np.log(len(ham_column) / corpus) # log P(ham)

In [31]:
# Compute likelihoods
alpha = 1
spam_likelihoods = np.zeros(len(global_vocab))
ham_likelihoods = np.zeros(len(global_vocab))

Ns = len(spam_vocab) # total number of words in spam
Nh = len(ham_vocab) # total number of words in ham

spam_word_count = Counter(spam_vocab)
ham_word_count = Counter(ham_vocab)

for word, index in indexed_vocab.items():
    Nws = spam_word_count.get(word, 0)
    Nwh = ham_word_count.get(word, 0)

    spam_likelihoods[index] = np.log((Nws + alpha) / (Ns + V * alpha))
    ham_likelihoods[index] = np.log((Nwh + alpha) / (Nh + V * alpha))

log_likelihoods = np.vstack([spam_likelihoods, ham_likelihoods])

# contructed a likelihood matrix of 2 by N 
# [
#   [spam_likelihoods],
#   [ham_likelihoods]
# ]

In [39]:
# save model to model.json
model = {
    "vocab": indexed_vocab,
    "log_priors": [float(prior_spam), float(prior_ham)],
    "log_likelihoods": log_likelihoods.tolist(),
}

with open("../model/model.json", "w") as f:
    json.dump(model, f, indent=2)

In [ ]:
# Prediction (sanity check)
def predict(email: str) -> str:
    cleaned = clean(email)

   # load our model
    with open("../model/model.json", "r") as model_f:
        model = json.load(model_f)

    # load model values
    vocab = model["vocab"]
    log_prior_s, log_prior_h = np.array(model["log_priors"])
    log_likelihoods = np.array(model["log_likelihoods"])

    # build a document vector, look up the index of each word in the cleaned email 
    # add 1 everytime that index is seen 
    doc_vector = np.zeros(len(vocab.values()))
    for word in cleaned:
        index = vocab.get(word)
        if index is None:
            continue
        doc_vector[index] += 1

    # compute the dot product of the document vector with the spam and ham log likelihoood vectors and
    # compute the argmax of them both, and return spam for argmax 1 and ham for argmax 2 
    log_post_spam = log_prior_s + (doc_vector @ log_likelihoods[0])
    log_post_ham = log_prior_h + (doc_vector @ log_likelihoods[1])

    argmax = np.argmax([log_post_spam, log_post_ham])
    return "spam" if argmax == 0 else "ham"


ham
spam


In [ ]:
ham_sample = """
Hi Luke,

We were unable to process your payment for your Personal Plan using Mastercard ending in 2867. To ensure you don't lose access to your valuable learning benefits, please update your payment details as soon as possible.

Don't miss out on what makes your Personal Plan so powerful:

✅ 26,000+ top-rated courses (including 1,300+ on AI)
✅ Hands-on labs & coding exercises with AI Role Plays
✅ 12,000+ certification prep courses for career advancement
✅ Personalized learning that fits any career path
Keep your learning momentum going → Update your payment method now in the Subscriptions tab of your account settings to maintain uninterrupted access to all these benefits.

Have questions? Check out our FAQ.

Keep growing,

The Udemy Team
"""

spam_sample = """
See it differently
Time away can change what you’re looking for and how things feel. Take another look and see what stands out now
"""

print(predict(ham_sample))
print(predict(spam_sample))